# 05 — Learning curve (Giai đoạn 5)

## Chuẩn bị môi trường và chạy thử

Trước khi chạy learning curve chính thức, em muốn chắc rằng môi trường trên Kaggle thực sự chạy được từ đầu đến cuối: lấy được mã
nguồn của đề tài, đọc đúng dữ liệu đã gán nhãn, có GPU, tải được ViSoBERT và huấn luyện được ít nhất một lượt nhỏ. Learning curve
cần huấn luyện nhiều lần (nhiều kích thước tập Train, mỗi kích thước ba seed), nên nếu đến lúc đó mới phát hiện thiếu thư viện hay
sai đường dẫn thì sẽ mất rất nhiều thời gian GPU. Vì vậy ở nhịp này em chỉ làm một lần chạy thử rất nhỏ, kết quả của nó không được
dùng cho bất kỳ phân tích nào.

Notebook được bật GPU và Internet, vì em cần clone mã nguồn từ GitHub và tải trọng số ViSoBERT từ Hugging Face.

Mã nguồn của đề tài nằm trong một repository riêng tư trên GitHub, nên em lưu token truy cập trong mục Secrets của Kaggle thay vì
ghi thẳng vào notebook. Sau khi clone, em đặt lại địa chỉ remote về dạng không chứa token, vì Kaggle lưu lại thư mục
`/kaggle/working` khi lưu phiên bản notebook và em không muốn token nằm trong đó. Lưu ý là bản trên GitHub chỉ có những gì em đã
push; ở bước này em chỉ cần các tiện ích chung trong `src.utils`.

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    # clone im lặng để token không xuất hiện trong log
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
# gỡ token khỏi .git/config (thư mục /kaggle/working được lưu cùng phiên bản notebook)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))

from src.utils.seeds import set_seed
from src.utils.kaggle_paths import is_kaggle
import yaml
set_seed(42)
commit = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()
print('Import src.utils: OK | Kaggle:', is_kaggle(), '| commit:', commit)
CFG = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'classification.yaml'), encoding='utf-8'))
print('Cấu hình ViSoBERT:', CFG['strategy_b_visobert'])

## Dữ liệu

Trên Kaggle em chỉ đưa lên hai tập dùng cho phát triển mô hình là Train và Validation. Hold-out, focused set và Lazada không có mặt
ở đây, đúng với thiết kế: các tập này bị khóa cho đến lần đánh giá cuối cùng nên không được xuất hiện trong môi trường huấn luyện.
Em kiểm tra lại kích thước và phân bố nhãn để chắc rằng đây đúng là phiên bản sau khi cân lại Validation (amendment v1.8):
Train 1.200 review, Validation 350 review với 22 review FEATURE.

In [ ]:
import pandas as pd
def find_data_dir(root='/kaggle/input', target='shopee_train.csv'):
    """Tìm thư mục chứa shopee_train.csv trong /kaggle/input (slug dataset có thể khác nhau)."""
    for dirpath, _, files in os.walk(root):
        if target in files:
            return dirpath
    found = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
    raise FileNotFoundError(f'Không tìm thấy {target} trong {root}. Các file hiện có:\n' +
                            ('\n'.join(found) if found else '(trống — chưa gắn dataset vào notebook)'))

DATA_DIR = find_data_dir()
print('DATA_DIR:', DATA_DIR)
train_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_validation.csv'), encoding='utf-8-sig')
order = ['BUG', 'FEATURE', 'UX', 'OTHER']
print(pd.DataFrame({'train': train_df['gold_label'].value_counts().reindex(order),
                    'validation': val_df['gold_label'].value_counts().reindex(order)}).to_string())
print('Train:', len(train_df), '| Validation:', len(val_df), '| FEATURE trong Validation:', int((val_df['gold_label'] == 'FEATURE').sum()))
assert len(train_df) == 1200 and len(val_df) == 350 and (val_df['gold_label'] == 'FEATURE').sum() == 22
assert not set(train_df['review_id']) & set(val_df['review_id']), 'Train và Validation trùng review'
print('Các file trong dataset:', sorted(os.listdir(DATA_DIR)))

## GPU và thư viện

Tiếp theo em kiểm tra GPU và thử tải tokenizer của ViSoBERT (`uitnlp/visobert`). Em chỉ tải tokenizer ở bước này và thử tách từ một
câu review tiếng Việt viết theo kiểu mạng xã hội, để xem tokenizer có xử lý được chữ có dấu, viết tắt và emoji hay không.

In [ ]:
import torch, transformers
from transformers import AutoTokenizer
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có GPU')
assert torch.cuda.is_available(), 'Chưa bật GPU cho notebook'
MODEL_NAME = CFG['strategy_b_visobert']['model_name']
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sample = 'app cứ bị đứng khi thanh toán, sp thì ok nhưng giao hơi lâu 😡'
enc = tokenizer(sample)
print('Câu mẫu:', sample)
print('Token:', tokenizer.convert_ids_to_tokens(enc['input_ids']))

## Chạy thử một lượt huấn luyện

Để kiểm tra toàn bộ đường đi từ dữ liệu đến dự đoán, em huấn luyện ViSoBERT cho tầng 1 (ACTIONABLE / NOT_ACTIONABLE), suy ra từ nhãn
4 lớp đúng như protocol: BUG, FEATURE và UX là ACTIONABLE, OTHER là NOT_ACTIONABLE. Em chỉ dùng 100 review đầu tiên của Train sau khi
xáo trộn với seed 42, huấn luyện đúng một epoch, rồi dự đoán trên 20 review Validation. Với lượng dữ liệu này mô hình chưa học được
gì đáng kể, và em cũng không dùng kết quả này để đánh giá; mục đích duy nhất là xác nhận việc tải mô hình, huấn luyện và dự đoán chạy
không lỗi, đồng thời đo thời gian để ước lượng chi phí GPU cho learning curve.

Vòng huấn luyện ở đây em viết thẳng trong cell vì nó chỉ dùng cho lần chạy thử; phần huấn luyện chính thức của các chiến lược sẽ nằm
trong `src/` như các bước trước.

In [ ]:
import time, numpy as np
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

ACTIONABLE = {'BUG', 'FEATURE', 'UX'}
to_tier1 = lambda lab: int(lab in ACTIONABLE)            # 1 = ACTIONABLE, 0 = NOT_ACTIONABLE
MAX_LEN, LR, BATCH = CFG['strategy_b_visobert']['max_len'], CFG['strategy_b_visobert']['lr'], 16
device = torch.device('cuda')

set_seed(42)
smoke_train = train_df.sample(n=100, random_state=42)
smoke_val = val_df.sample(n=20, random_state=42)

def batches(df, shuffle):
    data = list(zip(df['text'].astype(str), df['gold_label'].map(to_tier1)))
    def collate(b):
        enc = tokenizer([t for t, _ in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        enc['labels'] = torch.tensor([y for _, y in b])
        return enc
    return DataLoader(data, batch_size=BATCH, shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(42))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

torch.cuda.synchronize(); t0 = time.time()
model.train()
for b in batches(smoke_train, shuffle=True):
    b = {k: v.to(device) for k, v in b.items()}
    loss = model(**b).loss
    loss.backward(); opt.step(); opt.zero_grad()
torch.cuda.synchronize(); train_seconds = time.time() - t0
print(f'1 epoch trên {len(smoke_train)} review: {train_seconds:.1f} giây | loss cuối: {loss.item():.4f}')

model.eval(); preds = []
t0 = time.time()
with torch.no_grad():
    for b in batches(smoke_val, shuffle=False):
        b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
        preds += model(**b).logits.argmax(-1).tolist()
infer_seconds = time.time() - t0
names = {1: 'ACTIONABLE', 0: 'NOT_ACTIONABLE'}
out = smoke_val.assign(tier1_gold=smoke_val['gold_label'].map(lambda l: names[to_tier1(l)]),
                       tier1_pred=[names[p] for p in preds])
print(f'Dự đoán {len(smoke_val)} review Validation: {infer_seconds:.2f} giây')
print(out[['text', 'tier1_gold', 'tier1_pred']].head(8).assign(text=lambda d: d['text'].str[:70]).to_string(index=False))
print('Peak GPU memory:', round(torch.cuda.max_memory_allocated() / 2**30, 2), 'GB')

## Nhận xét về môi trường

Cell cuối cùng tổng hợp lại các lần kiểm tra ở trên và dùng thời gian đo được để ước lượng thời gian GPU cho learning curve. Ước
lượng này dựa trên giả định thời gian huấn luyện tăng gần tuyến tính theo số review, với các mốc 400, 800 và 1.200 review (Train
hiện có 1.200), mỗi mốc ba seed và bốn epoch theo cấu hình, cho cả hai tầng. Đây là cận trên, vì tầng 2 chỉ huấn luyện trên phần
actionable, ít hơn nhiều so với toàn bộ Train. Em sẽ ghi lại nhận xét cụ thể sau khi xem kết quả chạy trên Kaggle.

In [ ]:
per_review_epoch = train_seconds / len(smoke_train)
sizes, seeds, epochs, tiers = [400, 800, 1200], 3, CFG['strategy_b_visobert']['epochs'], 2
est = per_review_epoch * sum(sizes) * seeds * epochs * tiers
print(f'Môi trường: clone mã nguồn OK, dữ liệu OK (Train {len(train_df)}, Validation {len(val_df)}), '
      f'GPU {torch.cuda.get_device_name(0)}, tokenizer và mô hình ViSoBERT tải được, train + predict chạy không lỗi.')
print(f'Thời gian 1 epoch / 100 review: {train_seconds:.1f} giây (~{per_review_epoch * 1000:.0f} giây cho 1.000 review mỗi epoch).')
print(f'Ước lượng cận trên cho learning curve ViSoBERT (mốc {sizes}, {seeds} seed, {epochs} epoch, {tiers} tầng): '
      f'~{est / 60:.0f} phút GPU, chưa tính thời gian đánh giá trên Validation.')